# 00 — Open Model Distribution: Context and Leading Specification

**Primary specification:** open model distribution is specified by content addressing, cryptographic verification, and distributed replication.

This notebook anchors the `open-model-distribution` repository. It turns the two starting lab reports into a small executable model:

- `zanoga02`: **Content Addressing Specifies Model Distribution** — identity comes from content.
- `zanoga01`: **Distributed Replication Specifies Open AI Infrastructure** — availability comes from replication and independent paths.

The goal is not to build a full peer-to-peer system in notebook 00. The goal is to make the core constraints explicit, measurable, and reusable across later notebooks.


## 1. Context

Location-based model distribution says: retrieve a model from a URL, repository, bucket, mirror, or company-controlled platform.

Content-addressed model distribution says: retrieve the object with a known identity, then verify the bytes.

Distributed replication says: no single repository determines whether the object remains available.

The repo studies the engineering split:

\[
\text{identity} \neq \text{location}
\]

and then asks how identity, verification, replication, and independent paths produce resilient distribution.


In [ ]:
from pathlib import Path
import hashlib
import json
import math
import os
import shutil
import sys
import textwrap
import zipfile
from dataclasses import dataclass, asdict

import matplotlib.pyplot as plt
import pandas as pd

# Make notebook paths robust whether run from repo root or notebooks/.
CWD = Path.cwd()
if (CWD / "src").exists():
    REPO_ROOT = CWD
elif (CWD.parent / "src").exists():
    REPO_ROOT = CWD.parent
else:
    REPO_ROOT = Path("..").resolve()

FIGURES = REPO_ROOT / "figures"
RESULTS = REPO_ROOT / "results" / "00_context"
SITE_REPORTS = REPO_ROOT / "site" / "reports" / "00"

FIGURES.mkdir(parents=True, exist_ok=True)
RESULTS.mkdir(parents=True, exist_ok=True)
SITE_REPORTS.mkdir(parents=True, exist_ok=True)

SRC = REPO_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

try:
    from open_model_distribution.hashing import content_id, sha256_file
    from open_model_distribution.metrics import availability, distribution_score
except Exception:
    # Fallback definitions keep the notebook executable even before package install.
    def sha256_file(path, chunk_size=1024 * 1024):
        digest = hashlib.sha256()
        with Path(path).open("rb") as handle:
            for chunk in iter(lambda: handle.read(chunk_size), b""):
                digest.update(chunk)
        return digest.hexdigest()

    def content_id(path, algorithm="sha256"):
        if algorithm != "sha256":
            raise ValueError("Only sha256 is currently supported.")
        return f"sha256:{sha256_file(path)}"

    def availability(replication, independent_paths):
        return replication * independent_paths

    def distribution_score(content, verification, replication, paths):
        return content * verification * replication * paths

print(f"Repo root: {REPO_ROOT}")
print(f"Results:   {RESULTS}")


## 2. Leading specification

The primary repository statement is:

> **Open model distribution is specified by content addressing, cryptographic verification, and distributed replication.**

A useful first decomposition is:

\[
I = H(C)
\]

\[
V = \left(H(C_{\mathrm{received}}) = H(C_{\mathrm{published}})\right)
\]

\[
A \propto RP
\]

where:

- \(I\) is object identity.
- \(C\) is content.
- \(H\) is a cryptographic hash function.
- \(V\) is verification.
- \(A\) is availability.
- \(R\) is independent replication.
- \(P\) is independent network paths.


In [ ]:
@dataclass(frozen=True)
class DistributionVariables:
    identity: str
    content: str
    verification: str
    availability: str
    replication: str
    paths: str

variables = DistributionVariables(
    identity="I = canonical object identity derived from bytes",
    content="C = immutable model/software/data bytes",
    verification="V = cryptographic equality check against published identity",
    availability="A = probability the object remains retrievable",
    replication="R = independent copies across independent operators",
    paths="P = independent routes, mirrors, and transport protocols",
)

pd.DataFrame([asdict(variables)]).T.rename(columns={0: "meaning"})


## 3. Minimal content-addressed object

The smallest executable demonstration is a toy model artifact. The artifact's address is not its filename or folder. The address is the digest of the bytes.


In [ ]:
artifact_dir = RESULTS / "artifacts"
artifact_dir.mkdir(parents=True, exist_ok=True)

artifact_path = artifact_dir / "toy-model-card.txt"
artifact_text = """
model: toy-open-model
weights: placeholder
license: placeholder
created_for: open-model-distribution notebook 00
spec: content-addressed distribution
""".strip() + "\n"
artifact_path.write_text(artifact_text, encoding="utf-8")

cid = content_id(artifact_path)
sha = sha256_file(artifact_path)

manifest = {
    "name": "toy-open-model",
    "artifact": str(artifact_path.relative_to(REPO_ROOT)),
    "algorithm": "sha256",
    "digest": sha,
    "content_id": cid,
    "statement": "Content identity follows bytes, not location.",
}

manifest_path = RESULTS / "toy-model-manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")

manifest


## 4. Identity persists when location changes

If the same bytes move to another path, the content identity remains unchanged.


In [ ]:
mirror_dir = RESULTS / "mirror"
mirror_dir.mkdir(exist_ok=True)
mirror_path = mirror_dir / "renamed-object.bin"
shutil.copyfile(artifact_path, mirror_path)

identity_check = pd.DataFrame([
    {"location": str(artifact_path.relative_to(REPO_ROOT)), "content_id": content_id(artifact_path)},
    {"location": str(mirror_path.relative_to(REPO_ROOT)), "content_id": content_id(mirror_path)},
])
identity_check["same_identity"] = identity_check["content_id"].eq(cid)
identity_check


## 5. Verification catches changed content

If the bytes change, the identity changes. A location may look similar, but the content no longer verifies against the published digest.


In [ ]:
tampered_path = artifact_dir / "toy-model-card-tampered.txt"
tampered_path.write_text(artifact_text.replace("placeholder", "changed", 1), encoding="utf-8")

def verifies(path: Path, published_digest: str) -> bool:
    return sha256_file(path) == published_digest

verification_table = pd.DataFrame([
    {"artifact": "original", "path": str(artifact_path.relative_to(REPO_ROOT)), "digest": sha256_file(artifact_path), "verifies": verifies(artifact_path, sha)},
    {"artifact": "mirror", "path": str(mirror_path.relative_to(REPO_ROOT)), "digest": sha256_file(mirror_path), "verifies": verifies(mirror_path, sha)},
    {"artifact": "changed bytes", "path": str(tampered_path.relative_to(REPO_ROOT)), "digest": sha256_file(tampered_path), "verifies": verifies(tampered_path, sha)},
])
verification_table


## 6. Toy availability surface

Notebook 00 uses a deliberately simple availability score:

\[
A \propto RP
\]

This is not a production reliability model. It is a specification diagram: availability improves when replication and independent paths improve separately.


In [ ]:
rows = []
for R in range(1, 9):
    for P in range(1, 9):
        rows.append({"replicas_R": R, "paths_P": P, "availability_score_A": availability(R, P)})
availability_df = pd.DataFrame(rows)
availability_csv = RESULTS / "availability_surface.csv"
availability_df.to_csv(availability_csv, index=False)
availability_df.head(10)


In [ ]:
pivot = availability_df.pivot(index="paths_P", columns="replicas_R", values="availability_score_A")

fig, ax = plt.subplots(figsize=(7, 5))
im = ax.imshow(pivot.values, origin="lower", aspect="auto")
ax.set_xticks(range(len(pivot.columns)))
ax.set_xticklabels(pivot.columns)
ax.set_yticks(range(len(pivot.index)))
ax.set_yticklabels(pivot.index)
ax.set_xlabel("Independent replicas R")
ax.set_ylabel("Independent paths P")
ax.set_title("Toy availability score: A ∝ R P")
fig.colorbar(im, ax=ax, label="A score")
fig.tight_layout()

availability_fig = FIGURES / "00_availability_surface.png"
fig.savefig(availability_fig, dpi=180)
plt.show()

availability_fig


## 7. Distribution score

A second toy score combines object identity and network availability:

\[
D \propto CVRP
\]

For this notebook, \(C\) and \(V\) are binary gates:

- \(C = 1\) when the content is present.
- \(V = 1\) when the content verifies against the published identity.

If verification fails, the object should not enter the distribution graph.


In [ ]:
scenarios = pd.DataFrame([
    {"scenario": "central repository only", "C": 1, "V": 1, "R": 1, "P": 1},
    {"scenario": "verified mirror", "C": 1, "V": 1, "R": 2, "P": 2},
    {"scenario": "verified P2P swarm", "C": 1, "V": 1, "R": 6, "P": 5},
    {"scenario": "many copies, unverified", "C": 1, "V": 0, "R": 6, "P": 5},
    {"scenario": "verified metadata, missing content", "C": 0, "V": 1, "R": 6, "P": 5},
])

scenarios["D_score"] = scenarios.apply(
    lambda row: distribution_score(row.C, row.V, row.R, row.P), axis=1
)

scenarios_path = RESULTS / "distribution_scenarios.csv"
scenarios.to_csv(scenarios_path, index=False)
scenarios


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.barh(scenarios["scenario"], scenarios["D_score"])
ax.set_xlabel("Toy distribution score D ∝ C V R P")
ax.set_title("Verification gates distribution value")
fig.tight_layout()

score_fig = FIGURES / "00_distribution_score.png"
fig.savefig(score_fig, dpi=180)
plt.show()

score_fig


## 8. Report map

The first two reports split the primary specification into two reusable parts.


In [ ]:
report_map = pd.DataFrame([
    {
        "report": "zanoga02",
        "title": "Content Addressing Specifies Model Distribution",
        "primary_property": "identity and integrity",
        "equation": "I = H(C), V = H(received) = H(published)",
        "site_path": "/zanoga02/",
    },
    {
        "report": "zanoga01",
        "title": "Distributed Replication Specifies Open AI Infrastructure",
        "primary_property": "availability and resilience",
        "equation": "A ∝ R V P",
        "site_path": "/zanoga01/",
    },
])

report_map_path = RESULTS / "report_map.csv"
report_map.to_csv(report_map_path, index=False)
report_map


## 9. Notebook roadmap

The repo can now grow in small, testable steps.


In [ ]:
roadmap = pd.DataFrame([
    {"notebook": "00_context", "specification": "open model distribution", "question": "What are the primary variables?"},
    {"notebook": "07_content_addressing", "specification": "content identity", "question": "How do bytes become addresses?"},
    {"notebook": "13_cryptographic_verification", "specification": "integrity", "question": "How are received bytes verified?"},
    {"notebook": "17_chunking", "specification": "transfer units", "question": "How do large artifacts become movable parts?"},
    {"notebook": "23_replication", "specification": "availability", "question": "How many independent copies are enough?"},
    {"notebook": "29_peer_to_peer_networks", "specification": "routing", "question": "How are peers discovered and reached?"},
    {"notebook": "37_reproducibility", "specification": "release reconstruction", "question": "Can the same artifact be retrieved later?"},
    {"notebook": "43_reference_architecture", "specification": "system composition", "question": "How do the parts fit together?"},
])
roadmap_path = RESULTS / "notebook_roadmap.csv"
roadmap.to_csv(roadmap_path, index=False)
roadmap


## 10. Export a site stub for `labreports/zanoga/00.html`

Later notebooks can each receive a matching page. This cell writes a minimal static HTML page that can be replaced by the full site template when the repo is deployed.


In [ ]:
site_html = """<!doctype html>
<html lang=\"en\" data-theme=\"light\">
<head>
  <meta charset=\"utf-8\">
  <meta name=\"viewport\" content=\"width=device-width, initial-scale=1\">
  <title>00 Context | Open Model Distribution</title>
  <link rel=\"stylesheet\" href=\"/styles/site.css\">
  <script defer src=\"/js/dark-mode.js\"></script>
</head>
<body>
  <main class=\"site-main\">
    <article class=\"lab-report\">
      <section class=\"lab-hero\">
        <p class=\"lab-kicker\">open-model-distribution · notebook 00</p>
        <h1>Open Model Distribution: Context and Leading Specification</h1>
        <p class=\"lab-subtitle\">Open model distribution is specified by content addressing, cryptographic verification, and distributed replication.</p>
      </section>
      <section class=\"lab-note\">
        <h2>Core equations</h2>
        <p>Identity: \\(I = H(C)\\)</p>
        <p>Verification: \\(V = H(C_{received}) = H(C_{published})\\)</p>
        <p>Availability: \\(A \\propto RP\\)</p>
      </section>
    </article>
  </main>
</body>
</html>
"""

site_path = SITE_REPORTS / "00.html"
site_path.write_text(site_html, encoding="utf-8")
site_path


## 11. Save notebook manifest

This manifest records the artifacts produced by notebook 00.


In [ ]:
notebook_manifest = {
    "notebook": "00_context.ipynb",
    "title": "Open Model Distribution: Context and Leading Specification",
    "primary_specification": "open model distribution",
    "statement": "Open model distribution is specified by content addressing, cryptographic verification, and distributed replication.",
    "outputs": {
        "manifest": str(manifest_path.relative_to(REPO_ROOT)),
        "availability_csv": str(availability_csv.relative_to(REPO_ROOT)),
        "distribution_scenarios_csv": str(scenarios_path.relative_to(REPO_ROOT)),
        "report_map_csv": str(report_map_path.relative_to(REPO_ROOT)),
        "roadmap_csv": str(roadmap_path.relative_to(REPO_ROOT)),
        "availability_figure": str(availability_fig.relative_to(REPO_ROOT)),
        "distribution_score_figure": str(score_fig.relative_to(REPO_ROOT)),
        "site_stub": str(site_path.relative_to(REPO_ROOT)),
    },
}

notebook_manifest_path = RESULTS / "00_context_manifest.json"
notebook_manifest_path.write_text(json.dumps(notebook_manifest, indent=2), encoding="utf-8")
notebook_manifest


## 12. Download artifacts

Run the final cell to package the notebook outputs for download.


In [ ]:
zip_path = REPO_ROOT / "results" / "00_context_artifacts.zip"
paths_to_include = [
    RESULTS,
    FIGURES / "00_availability_surface.png",
    FIGURES / "00_distribution_score.png",
    SITE_REPORTS / "00.html",
]

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for item in paths_to_include:
        item = Path(item)
        if item.is_dir():
            for path in item.rglob("*"):
                if path.is_file():
                    zf.write(path, path.relative_to(REPO_ROOT))
        elif item.exists():
            zf.write(item, item.relative_to(REPO_ROOT))

print(f"Created: {zip_path}")
print(f"Size: {zip_path.stat().st_size:,} bytes")
